## Non Linear Transformer

In [ ]:
import pandas as pd
import numpy as np

### Annual Lag Appending for K1 Characteristics

For each such characteristic, the rank-normalised current observation is augmented with five annual lags at months $\{12, 24, 36, 48, 60\}$ to get six observations per K1 characteristic. The resulting flat input vector $\bar{x}_{i,t} \in \mathbb{R}^{K_0 + 6K_1}$ is constructed exclusively from information available at or before month $t$.

In [ ]:
LAG_MONTHS = [0, 12, 24, 36, 48, 60]


def append_annual_lags( panel: pd.DataFrame, k1_cols: list[str], date_col: str = "date", firm_col: str = "gvkey") -> pd.DataFrame: 

    panel = panel.sort_values([firm_col, date_col]).copy()
    for col in k1_cols:
        for lag in LAG_MONTHS[1:]:
            lag_col = f"{col}_lag{lag}"
            shifted = (panel[[firm_col, date_col, col]].copy().assign(**{date_col: lambda df: df[date_col] + pd.DateOffset(months=lag)}).rename(columns={col: lag_col})
            )
            panel = panel.merge(shifted, on=[firm_col, date_col], how="left")
    return panel


def build_flat_input_vector(panel: pd.DataFrame, k0_cols: list[str], k1_cols: list[str], date_col: str = "date", 
                            firm_col: str = "permno") -> tuple[np.ndarray, list[str], pd.DataFrame]:

    ordered = list(k0_cols)
    for col in k1_cols:
        ordered.append(col)
        for lag in LAG_MONTHS[1:]:
            ordered.append(f"{col}_lag{lag}")
    available = [c for c in ordered if c in panel.columns]
    X = panel[available].to_numpy(dtype=np.float32)
    meta = panel[[firm_col, date_col]].reset_index(drop=True)
    expected = len(k0_cols) + 6 * len(k1_cols)
    print(
        f"Flat input vector assembled: {X.shape[0]:,} observations, "
        f"{X.shape[1]} features (expected K0 + 6*K1 = {expected})."
    )
    return X, available, meta